# Data Explorer: PubSub OHLCV

Analyze PubSub crawl data with OHLCV candlestick charts.

In [19]:
import os
import glob
import pandas as pd
import numpy as np
import pyarrow.feather as feather
import matplotlib.pyplot as plt
# import mplfinance as mpf  # Uncomment if installed

## Configuration

In [20]:
# Config
PUBSUB_DIR = "./data/pubsub"

# Stock to analyze
STOCK = "41I1G3000"

# Time interval for OHLCV (e.g., '1min', '5min', '15min', '1h')
TIME_INTERVAL = '1min'

## Load Data

In [21]:
# Find latest date with data
def get_latest_date(data_dir, stock):
    stock_dir = os.path.join(data_dir, stock)
    if not os.path.exists(stock_dir):
        return None
    files = glob.glob(os.path.join(stock_dir, "*.fea"))
    if not files:
        return None
    dates = [os.path.splitext(os.path.basename(f))[0] for f in files]
    return max(dates)

latest_date = get_latest_date(PUBSUB_DIR, STOCK)
print(f"Latest date: {latest_date}")

Latest date: 2026-03-12


In [22]:
# Load PubSub data
if latest_date:
    pubsub_path = os.path.join(PUBSUB_DIR, STOCK, f"{latest_date}.fea")
    df = pd.read_feather(pubsub_path)
    print(f"Loaded {len(df)} rows from {pubsub_path}")
else:
    df = None
    print("No data found")

Loaded 479843 rows from ./data/pubsub/41I1G3000/2026-03-12.fea


In [23]:
df.head()

,tick,stk,ceil,floor,adj,b3,nsb3,b2,nsb2,b1,...,nss2,s3,nss3,ttqt,high,low,fb,fs,fr,msg_id
0,2026-03-12T09:24:51.125225,41I1G3000,2.0148,1.7512,1.883,0.18486,3,1.8487,5,1.8488,...,1,0.18496,24,53083,1.8695,1.8467,2296,2816,0,b'1773282291124-0'
1,2026-03-12T09:24:51.136650,41I1G3000,2.0148,1.7512,1.883,0.18485,2,1.8486,4,1.8487,...,9,0.18495,1,53085,1.8695,1.8467,2296,2816,0,b'1773282291125-0'
2,2026-03-12T09:24:51.136769,41I1G3000,2.0148,1.7512,1.883,0.18486,3,1.8487,5,1.8488,...,1,0.18496,24,53083,1.8695,1.8467,2296,2816,0,b'1773282291128-0'
3,2026-03-12T09:24:51.136787,41I1G3000,2.0148,1.7512,1.883,0.18486,4,1.8487,5,1.8488,...,1,0.18496,24,53083,1.8695,1.8467,2296,2816,0,b'1773282291129-0'
4,2026-03-12T09:24:51.136803,41I1G3000,2.0148,1.7512,1.883,0.18486,4,1.8487,5,1.8488,...,1,0.18496,24,53085,1.8695,1.8467,2296,2816,0,b'1773282291130-0'


## OHLCV Converter

In [24]:
import pandas as pd
import datetime

def tick_to_ohlcv(df, interval='1min', price_col='match_price', volume_col='total_volume'):
    """
    Convert tick data to OHLCV candlestick data, filtering valid market hours.
    """
    if df is None or df.empty:
        return None
    
    df = df.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    tick_times = df['timestamp'].dt.time
    
    # Define valid market windows
    # morning_valid = tick_times <= datetime.time(11, 30, 0)
    # afternoon_valid = (tick_times >= datetime.time(13, 0, 0)) & (tick_times <= datetime.time(14, 30, 0))
    
    # Keep only rows that fall in the morning OR afternoon session
    # df = df[morning_valid | afternoon_valid]
    # ---------------------------------------------------------
    
    df = df.set_index('timestamp')
    
    # pandas .ohlc() automatically creates 'open', 'high', 'low', 'close'
    ohlcv = df[price_col].resample(interval).ohlc()
    ohlcv['volume'] = df[volume_col].resample(interval).sum()
    
    # Drop periods with no trading activity
    ohlcv = ohlcv.dropna(subset=['open']).reset_index()
    
    return ohlcv

In [25]:
ohlcv = tick_to_ohlcv(df, interval=TIME_INTERVAL)

ohlcv.head()

KeyError: 'timestamp'

## Visualize OHLCV

In [ ]:

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Convert to OHLCV
if df is not None:
    ohlcv = tick_to_ohlcv(df, interval=TIME_INTERVAL)
    print(f"OHLCV - {TIME_INTERVAL}: {len(ohlcv)} candles")
    
    # Determine Volume bar colors (Green for bullish, Red for bearish)
    volume_colors = ['#26a69a' if row['close'] >= row['open'] else '#ef5350' for _, row in ohlcv.iterrows()]

    # 2. Create interactive subplots (Row 1 = Candlestick, Row 2 = Volume)
    fig = make_subplots(
        rows=2, cols=1, 
        shared_xaxes=True, 
        vertical_spacing=0.03, 
        subplot_titles=(f'{STOCK} Price', 'Volume'), 
        row_width=[0.2, 0.7]  # 70% height for price, 20% for volume
    )

    # 3. Add Candlestick Trace
    fig.add_trace(
        go.Candlestick(
            x=ohlcv['tick'],
            open=ohlcv['open'],
            high=ohlcv['high'],
            low=ohlcv['low'],
            close=ohlcv['close'],
            name='Price',
            increasing_line_color='#26a69a', # Custom green
            decreasing_line_color='#ef5350'  # Custom red
        ),
        row=1, col=1
    )

    # 4. Add Volume Trace
    fig.add_trace(
        go.Bar(
            x=ohlcv['tick'], 
            y=ohlcv['volume'], 
            name='Volume',
            marker_color=volume_colors
        ),
        row=2, col=1
    )

    # 5. Format layout
    fig.update_layout(
        title=f"{STOCK} - {TIME_INTERVAL} Interactive OHLCV Chart",
        xaxis_rangeslider_visible=False, # Hides the redundant rangeslider at the bottom
        height=700,
        template='plotly_white',
        showlegend=False
    )
    
    # Tweak y-axis to not start at 0 for price
    fig.update_yaxes(title_text="Price", row=1, col=1)
    fig.update_yaxes(title_text="Volume", row=2, col=1)

    # Add this right before fig.show() to hide the lunch gap!
    fig.update_xaxes(
        rangebreaks=[
            dict(bounds=["11:30", "13:00"], pattern="hour"), # Hides the lunch break
            dict(bounds=["14:30", "09:00"], pattern="hour")  # Hides overnight gap (if plotting multiple days)
        ]
    )

    fig.show()
else:
    print("No OHLCV data to plot")

OHLCV - 1min: 213 candles
